# Spotify Music Popularity Prediction

**Name**: Ivan Hughes

**Website Link**: (your website link)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'
import plotly.io as pio

## Introduction

### Dataset
The dataset contains 114,000 Spotify songs from `music_tracks.csv`, 
along with artist information from `artists.csv` (1,162,095 artists). 
Together, these files provide a comprehensive view of songs on Spotify, 
including audio features, metadata, and artist characteristics.

### Central Question
**What audio features and artist characteristics best predict a song's 
popularity on Spotify?**

Understanding what makes songs popular has real-world implications for 
artists, producers, and record labels. By finding key predictors of 
popularity, artists can make more informed decisions about their music, 
streaming platforms can improve their recommendation systems, and record 
labels can better identify potential hits before release.

### Relevant Columns

| Column | Description |
|--------|-------------|
| `popularity` | Song's popularity score (0-100) |
| `danceability` | How suitable for dancing (0-1) |
| `energy` | Intensity and activity (0-1) |
| `valence` | Musical positiveness (0-1) |
| `loudness` | Overall loudness in decibels |
| `speechiness` | Presence of spoken words (0-1) |
| `acousticness` | Whether acoustic (0-1) |
| `instrumentalness` | Whether instrumental (0-1) |
| `liveness` | Live performance detection (0-1) |
| `tempo` | Beats per minute (BPM) |
| `explicit` | Whether explicit (True/False) |
| `track_genre` | Genre of the song |
| `artists` | Artist name(s) |
| `artist_popularity` | Artist's popularity score (0-100, from artists.csv) |
| `artist_followers` | Number of artist followers (from artists.csv) |

## Data Cleaning and Exploratory Data Analysis

In [2]:
pio.renderers.default = 'iframe'
df = pd.read_csv('Data/music_tracks.csv')
artists = pd.read_csv('data/artists.csv')
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,NaN,4,acoustic


In [3]:
# ============================================================
# STEP 2: DATA CLEANING
# ============================================================

df_clean = df.copy()

# 1. Drop unnamed index column
df_clean = df_clean.drop(columns=['Unnamed: 0'], errors='ignore')

# 2. Drop rows with missing names
df_clean = df_clean.dropna(subset=['artists', 'album_name', 'track_name'])

# 3. Convert release_date to datetime
df_clean['release_date'] = pd.to_datetime(
    df_clean['release_date'], 
    format='mixed',
    errors='coerce'
)
df_clean['release_year'] = df_clean['release_date'].dt.year

# 4. Convert explicit to boolean
df_clean['explicit'] = df_clean['explicit'].astype(bool)

# 5. Remove duplicate tracks
df_clean = df_clean.drop_duplicates(subset=['track_id'])

# 6. Extract first artist
df_clean['first_artist'] = df_clean['artists'].str.split(';').str[0]

# 7. Merge with artist data
if 'artist_popularity' not in df_clean.columns:
    artists_clean = artists[['name', 'popularity', 'followers']].rename(columns={
        'popularity': 'artist_popularity',
        'followers': 'artist_followers'
    })
    df_clean = df_clean.merge(
        artists_clean,
        left_on='first_artist',
        right_on='name',
        how='left'
    )

df = df_clean.copy()

# Show results
print(f"Shape before cleaning: {df.shape}")
print(f"Shape after cleaning:  {df_clean.shape}")
print(f"\nMissing values after cleaning:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"\nCleaned DataFrame:")
display(df_clean.head())

Shape before cleaning: (98862, 26)
Shape after cleaning:  (98862, 26)

Missing values after cleaning:
tempo                19360
name                  3498
artist_popularity     3498
artist_followers      3498
dtype: int64

Cleaned DataFrame:


,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,energy,...,liveness,valence,tempo,time_signature,track_genre,release_year,first_artist,name,artist_popularity,artist_followers
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974-01-01,False,0.676,0.4610,...,0.3580,0.715,87.917,4,acoustic,1974,Gen Hoshino,Gen Hoshino,66.0,852637.0
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04-01,False,0.420,0.1660,...,0.1010,0.267,77.489,4,acoustic,1995,Ben Woodward,Ben Woodward,53.0,11874.0
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973-01-01,False,0.438,0.3590,...,0.1170,0.120,76.332,4,acoustic,1973,Ingrid Michaelson,Ingrid Michaelson,68.0,722496.0
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.266,0.0596,...,0.1320,0.143,181.740,3,acoustic,2018,Kina Grannis,Kina Grannis,71.0,438860.0
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.618,0.4430,...,0.0829,0.167,NaN,4,acoustic,2017,Chord Overstreet,Chord Overstreet,70.0,99345.0


## Step 3: Assessment of Missingness

In [4]:
print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

# Check missing values in each column
missing_counts = df.isnull().sum()
missing_pcts = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pcts
})

# Only show columns with missing values
print(missing_df[missing_df['Missing Count'] > 0])
print(f"\nTotal rows: {len(df)}")

MISSING VALUE ANALYSIS
                   Missing Count  Missing %
tempo                      19360      19.58
name                        3498       3.54
artist_popularity           3498       3.54
artist_followers            3498       3.54

Total rows: 98862


==============================================================

NMAR Analysis:

I don't think any column in this dataset is NMAR (Missing Not 
At Random). The tempo column has the most significant missingness 
at 19.4% (22,114 out of 114,000 rows).

I believe tempo is MAR (Missing At Random) and not NMAR 
because its missingness seems to depend on danceability 
(p < 0.001) rather than on the tempo values themselves. 

This makes sense because songs with lower danceability 
like ambient, classical, or spoken word tracks may have 
irregular rhythms that make it difficult for Spotify's audio 
analysis system to detect a clear BPM value, causing tempo 
to go unrecorded.

If tempo were NMAR, the missingness would depend on the actual 
tempo value itself (extremely fast or slow songs being 
harder to measure). To further investigate, it would be helpful 
to obtain data on the audio analysis confidence score that 
Spotify uses internally, which would help explain why certain 
tracks fail to have tempo recorded.

In [5]:
# ============================================================
# STEP 3: MISSINGNESS DEPENDENCY
# ============================================================

# Create missingness indicator for tempo
df['tempo_missing'] = df['tempo'].isna()

# ============================================================
# TEST 1: Does tempo missingness DEPEND on popularity?
# ============================================================

# Get popularity for missing and not missing groups
pop_missing = df[df['tempo_missing']]['popularity'].dropna().values
pop_not_missing = df[~df['tempo_missing']]['popularity'].dropna().values

# Observed test statistic
observed_diff_pop = abs(pop_missing.mean() - pop_not_missing.mean())
print(f"Mean popularity when tempo IS missing:     {pop_missing.mean():.2f}")
print(f"Mean popularity when tempo is NOT missing: {pop_not_missing.mean():.2f}")
print(f"Observed difference:                       {observed_diff_pop:.4f}")

# Permutation test
all_pop = np.concatenate([pop_missing, pop_not_missing])
n_missing = len(pop_missing)
count_pop = 0

for i in range(500):
    shuffled = np.random.permutation(all_pop)
    diff = abs(shuffled[:n_missing].mean() - shuffled[n_missing:].mean())
    if diff >= observed_diff_pop:
        count_pop += 1

p_value_pop = float(count_pop / 500)
print(f"P-value: {p_value_pop:.4f}")
print(f"Conclusion: tempo missingness does NOT depend on popularity")

# ============================================================
# TEST 2: Does tempo missingness DEPEND on danceability?
# ============================================================

# Get danceability for missing and not missing groups
dance_missing = df[df['tempo_missing']]['danceability'].dropna().values
dance_not_missing = df[~df['tempo_missing']]['danceability'].dropna().values

# Observed test statistic
observed_diff_dance = abs(dance_missing.mean() - dance_not_missing.mean())
print(f"\nMean danceability when tempo IS missing:     {dance_missing.mean():.2f}")
print(f"Mean danceability when tempo is NOT missing: {dance_not_missing.mean():.2f}")
print(f"Observed difference:                         {observed_diff_dance:.4f}")

# Permutation test
all_dance = np.concatenate([dance_missing, dance_not_missing])
n_missing_dance = len(dance_missing)
count_dance = 0

for i in range(500):
    shuffled = np.random.permutation(all_dance)
    diff = abs(shuffled[:n_missing_dance].mean() - shuffled[n_missing_dance:].mean())
    if diff >= observed_diff_dance:
        count_dance += 1

p_value_dance = float(count_dance / 500)
print(f"P-value: {p_value_dance:.4f}")
print(f"Conclusion: tempo missingness DOES depend on danceability")

# ============================================================
# SUMMARY
# ============================================================

print(f"\nMissingness Summary:")
print(f"  Popularity   p-value: {p_value_pop:.4f}   → does NOT depend on popularity")
print(f"  Danceability p-value: {p_value_dance:.4f}   → DOES depend on danceability")

Mean popularity when tempo IS missing:     33.39
Mean popularity when tempo is NOT missing: 33.50
Observed difference:                       0.1021
P-value: 0.4820
Conclusion: tempo missingness does NOT depend on popularity

Mean danceability when tempo IS missing:     0.55
Mean danceability when tempo is NOT missing: 0.57
Observed difference:                         0.0136
P-value: 0.0000
Conclusion: tempo missingness DOES depend on danceability

Missingness Summary:
  Popularity   p-value: 0.4820   → does NOT depend on popularity
  Danceability p-value: 0.0000   → DOES depend on danceability


In [6]:

# Create readable label
df['tempo_missing_label'] = df['tempo_missing'].map({
    True: 'Tempo Missing', 
    False: 'Tempo Not Missing'
})

# Plot distribution of DANCEABILITY by missingness
# (since danceability is what tempo missingness depends on)
fig = px.histogram(
    df,
    x='danceability',
    color='tempo_missing_label',
    barmode='overlay',
    opacity=0.7,
    nbins=50,
    title='Distribution of Danceability by Tempo Missingness',
    labels={
        'danceability': 'Danceability',
        'tempo_missing_label': 'Tempo Missingness'
    },
    color_discrete_map={
        'Tempo Missing': '#EF553B',
        'Tempo Not Missing': '#1DB954'
    }
)

fig.update_layout(
    xaxis_title='Danceability',
    yaxis_title='Count',
    legend_title='Tempo Missingness'
)

fig.show()

# Save for website
fig.write_html('missingness_plot.html')

## Step 4: Hypothesis Testing

## Question
Do explicit songs have significantly higher popularity 
than non-explicit songs on Spotify?

## Null Hypothesis
The distribution of popularity is the same for explicit 
and non-explicit songs. Any observed difference is due 
to random chance.

## Alternative Hypothesis
Explicit songs have a higher mean popularity than 
non-explicit songs.

## Test Statistic
Absolute difference in mean popularity between explicit 
and non-explicit songs.

WHY: We chose absolute difference in means because:
- Popularity is a quantitative variable (0-100 scale)
- We want to detect differences in the average popularity
- Absolute value because we're testing if they're different
  (two-sided), not specifically which direction

## Significance Level
α = 0.05 (5%)

## Results
- Explicit songs mean popularity:     36.45
- Non-explicit songs mean popularity: 32.94
- Observed difference in means:       3.5163
- Number of permutations:             500
- P-value:                            < 0.001

## Conclusion
Since p-value < 0.001 < α = 0.05, we reject the null 
hypothesis. The data suggests that explicit songs tend 
to have higher popularity than non-explicit songs on 
Spotify (36.45 vs 32.94 on average).

However, we cannot conclude that explicit content 
CAUSES higher popularity. Other confounding factors 
(such as genre - explicit songs are more common in 
rap/hip-hop which tends to be more popular) may 
explain this difference.

In [7]:
# Create visualization for hypothesis test
fig = px.histogram(
    df,
    x='popularity',
    color=df['explicit'].map({True: 'Explicit', False: 'Non-Explicit'}),
    barmode='overlay',
    opacity=0.7,
    nbins=50,
    title='Distribution of Popularity: Explicit vs Non-Explicit Songs',
    labels={
        'popularity': 'Popularity Score (0-100)',
        'color': 'Song Type'
    },
    color_discrete_map={
        'Explicit': '#EF553B',
        'Non-Explicit': '#1DB954'
    }
)

fig.update_layout(
    xaxis_title='Popularity Score (0-100)',
    yaxis_title='Count',
    legend_title='Song Type'
)

fig.show()
fig.write_html('hypothesis_test_plot.html')

## Step 5: Framing a Prediction Problem

#### Prediction Problem
I aim to predict the **popularity** of a Spotify song based on 
its audio features and metadata.

#### Type
This is a **regression** problem since popularity is a 
continuous numerical value ranging from 0 to 100.

#### Response Variable
**popularity** - I chose this because:
- It is a direct measure of a song's success on Spotify
- It is a continuous numerical value, making it suitable 
  for regression
- Understanding what makes a song popular is valuable 
  for artists, producers, and record labels

#### Evaluation Metric
I will use **RMSE (Root Mean Squared Error)** because:
- It is in the same units as popularity (0-100 scale), 
  making it easy to interpret
- It penalizes large errors more than small ones, which 
  is important since we want to avoid large mispredictions
- R² will also be reported to show how much variance 
  our model explains

#### Why not other metrics?
- **MAE:** Less sensitive to large errors, which we care about
- **MSE:** Same as RMSE but harder to interpret (squared units)
- **R²:** Useful but doesn't tell us the actual prediction error 
  in interpretable units

#### Features Available at Time of Prediction
The following features would be known BEFORE a song is released 
or immediately after upload to Spotify:

Can use (known at time of prediction):
- danceability    (audio feature, measured immediately)
- energy          (audio feature, measured immediately)
- valence         (audio feature, measured immediately)
- loudness        (audio feature, measured immediately)
- speechiness     (audio feature, measured immediately)
- acousticness    (audio feature, measured immediately)
- instrumentalness(audio feature, measured immediately)
- liveness        (audio feature, measured immediately)
- tempo           (audio feature, measured immediately)
- explicit        (known at upload)
- track_genre     (known at upload)
- release_date    (known at upload)

Cannot use (not known at time of prediction):
- popularity itself (this is what we're predicting!)
- Any features derived from popularity

For website:
Framing a Prediction Problem

Prediction Problem:
Predict the popularity score (0-100) of a Spotify song 
based on its audio features and metadata.

Type: Regression
- Popularity is a continuous numerical value (0-100)
- We are not classifying into categories

Response Variable: popularity
- Directly measures a song's success on Spotify
- Valuable for artists and producers to understand 
  what makes a song popular

Evaluation Metric: RMSE (Root Mean Squared Error)
- Interpretable: same units as popularity (0-100)
- Penalizes large errors more than small ones
- Better than MAE for catching large mispredictions
- Better than MSE because it's in interpretable units

Time of Prediction Justification:
All features used (danceability, energy, valence, loudness, 
speechiness, acousticness, instrumentalness, liveness, 
explicit, track_genre) are audio features measured by 
Spotify immediately when a song is uploaded. This means 
they would ALL be known at the time of prediction, making 
them valid features to include in our model.

## Step 6: Baseline Model

In [8]:
# ============================================================
# STEP 6: BASELINE MODEL
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

# ============================================================
# FEATURES
# ============================================================

# Quantitative (2): danceability, energy
#   - danceability: strong indicator of song style/appeal
#   - energy: captures intensity which affects popularity
# Nominal (2): explicit, track_genre
#   - explicit: shown in hypothesis test to affect popularity
#   - track_genre: different genres have different popularity levels

# Prepare data
df_model = df[['danceability', 'energy', 
               'explicit', 'track_genre', 
               'popularity']].dropna()

X = df_model[['danceability', 'energy', 'explicit', 'track_genre']]
y = df_model['popularity']

# Train/test split (random_state=42 reused in Step 7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")

# ============================================================
# BUILD PIPELINE
# ============================================================

# Quantitative: pass through as-is (already 0-1 scale)
# Nominal: OneHotEncoder

preprocessor = ColumnTransformer([
    ('quant', 'passthrough', ['danceability', 'energy']),
    ('nom', OneHotEncoder(handle_unknown='ignore', 
                         sparse_output=False), 
     ['explicit', 'track_genre'])
])

baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# ============================================================
# TRAIN AND EVALUATE
# ============================================================

baseline_pipeline.fit(X_train, y_train)

# Training performance
y_pred_train = baseline_pipeline.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

# Test performance
y_pred_test = baseline_pipeline.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

print(f"Training RMSE: {train_rmse:.4f}  |  Training R²: {train_r2:.4f}")
print(f"Test RMSE:     {test_rmse:.4f}  |  Test R²:     {test_r2:.4f}")
print(f"Generalization Gap - RMSE: {abs(train_rmse - test_rmse):.4f}, R²: {abs(train_r2 - test_r2):.4f}")

# ============================================================
# MODEL ASSESSMENT
# ============================================================

print(f"""
Baseline Model: Linear Regression
  Features: danceability, energy (quantitative)
            explicit, track_genre (nominal, one-hot encoded)

  Test RMSE: {test_rmse:.4f} (~{test_rmse:.1f} points off on 0-100 scale)
  Test R²:   {test_r2:.4f} (explains {test_r2*100:.1f}% of variance)

This model is not particularly good because:
  1. Linear Regression assumes linear relationships
  2. Only 4 features used out of many available
  3. R² of {test_r2:.2f} suggests many other factors affect popularity
""")

# Save for Step 7 comparison
baseline_test_rmse = test_rmse
baseline_test_r2 = test_r2

Training rows: 79089
Testing rows:  19773
Training RMSE: 16.8585  |  Training R²: 0.3265
Test RMSE:     17.2096  |  Test R²:     0.3120
Generalization Gap - RMSE: 0.3511, R²: 0.0145

Baseline Model: Linear Regression
  Features: danceability, energy (quantitative)
            explicit, track_genre (nominal, one-hot encoded)

  Test RMSE: 17.2096 (~17.2 points off on 0-100 scale)
  Test R²:   0.3120 (explains 31.2% of variance)

This model is not particularly good because:
  1. Linear Regression assumes linear relationships
  2. Only 4 features used out of many available
  3. R² of 0.31 suggests many other factors affect popularity



Step 6: Baseline Model

Model: Linear Regression

Features (4 total):

Quantitative (2):
- danceability (0-1 scale, left as-is)
- energy (0-1 scale, left as-is)

Nominal (2):
- explicit (OneHotEncoded: True/False → 2 binary columns)
- track_genre (OneHotEncoded: multiple binary columns, 
  one per genre)

Performance:
- Training RMSE: 19.26
- Test RMSE:     19.16
- Training R²:   0.2558
- Test R²:       0.2561

Generalization:
The training and test performance are nearly identical 
(RMSE difference: 0.10, R² difference: 0.0003), which 
means the model generalizes well to unseen data and is 
not overfitting.

Is this a good model?
No - the baseline model is not particularly good:
- RMSE of 19.16 means predictions are off by ~19 points 
  on a 0-100 scale, which is a large error
- R² of 0.2561 means the model only explains 25.6% of 
  the variance in popularity
- This is expected for a baseline - Linear Regression 
  assumes linear relationships which may not exist between 
  audio features and popularity
- Step 7 will improve on this by adding more features 
  and using a more powerful non-linear algorithm

In [9]:
# ============================================================
# HYPOTHESIS 2: Genre & Tempo Distribution
# ============================================================

# Get 3 genres to compare
genre_counts = df['track_genre'].value_counts()
top_genres = genre_counts.head(3).index.tolist()

print(f"Genres selected: {top_genres}")
print(f"Songs per genre: 1000 each")

# Filter to these genres (drop missing tempo)
df_genres = df[df['track_genre'].isin(top_genres)][['track_genre', 'tempo']].dropna()
print(f"Total songs after dropping missing tempo: {len(df_genres)}")

# Tempo statistics by genre
print("\nTempo statistics by genre:")
for genre in top_genres:
    genre_tempos = df_genres[df_genres['track_genre'] == genre]['tempo'].values
    print(f"  {genre}: mean={genre_tempos.mean():.2f}, std={genre_tempos.std():.2f}, n={len(genre_tempos)}")

# ============================================================
# PERMUTATION TEST
# ============================================================

# Test statistic: average absolute difference in mean tempo
# between all pairs of genres
def calculate_genre_tempo_diff(data, genres_list):
    diffs = []
    for i, genre1 in enumerate(genres_list):
        for genre2 in genres_list[i+1:]:
            tempo1 = data[data['track_genre'] == genre1]['tempo'].mean()
            tempo2 = data[data['track_genre'] == genre2]['tempo'].mean()
            diffs.append(abs(tempo1 - tempo2))
    return np.mean(diffs)

observed_tempo_diff = calculate_genre_tempo_diff(df_genres, top_genres)
print(f"\nObserved test statistic: {observed_tempo_diff:.4f}")

# Permutation test (1000 reps)
count = 0
for i in range(1000):
    df_shuffled = df_genres.copy()
    df_shuffled['track_genre'] = np.random.permutation(df_shuffled['track_genre'].values)
    shuffled_diff = calculate_genre_tempo_diff(df_shuffled, top_genres)
    if shuffled_diff >= observed_tempo_diff:
        count += 1

p_value_h2 = float(count / 1000)
print(f"P-value: {p_value_h2:.4f}")

if p_value_h2 < 0.05:
    print("Conclusion: REJECT null - genres have significantly different tempo distributions")
else:
    print("Conclusion: FAIL TO REJECT null - no significant difference in tempo across genres")

Genres selected: ['honky-tonk', 'anime', 'sleep']
Songs per genre: 1000 each
Total songs after dropping missing tempo: 2901

Tempo statistics by genre:
  honky-tonk: mean=116.00, std=29.90, n=932
  anime: mean=124.98, std=32.88, n=1084
  sleep: mean=88.81, std=40.80, n=885

Observed test statistic: 24.1151
P-value: 0.0000
Conclusion: REJECT null - genres have significantly different tempo distributions


## Step 7: Final Model

In [10]:
# ============================================================
# MERGE ARTIST DATA WITH TRACKS
# ============================================================

import pandas as pd

print("=" * 60)
print("MERGING ARTIST DATA")
print("=" * 60)

# Load artists data
artists = pd.read_csv('data/artists.csv')

print("Artists DataFrame:")
print(f"  Shape: {artists.shape}")
print(f"  Columns: {artists.columns.tolist()}")
print(f"\nFirst few rows:")
print(artists.head())

print("\n" + "=" * 60)
print("Tracks DataFrame:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\nFirst few rows of relevant columns:")
print(df[['track_id', 'artists', 'track_name']].head())

MERGING ARTIST DATA
Artists DataFrame:
  Shape: (1162095, 5)
  Columns: ['id', 'followers', 'genres', 'name', 'popularity']

First few rows:
                       id  followers genres  \
0  0DheY5irMjBUeLybbCUEZ2        0.0     []   
1  0DlhY15l3wsrnlfGio2bjU        5.0     []   
2  0DmRESX2JknGPQyO15yxg7        0.0     []   
3  0DmhnbHjm1qw6NCYPeZNgJ        0.0     []   
4  0Dn11fWM7vHQ3rinvWEl4E        2.0     []   

                                             name  popularity  
0  Armid & Amir Zare Pashai feat. Sara Rouzbehani           0  
1                                     ปูนา ภาวิณี           0  
2                                           Sadaa           0  
3                                       Tra'gruda           0  
4                          Ioannis Panoutsopoulos           0  

Tracks DataFrame:
  Shape: (98862, 28)
  Columns: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'release_date', 'explicit', 'danceability', 'energy', 'key',

In [11]:
# ============================================================
# FINAL MODEL
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, QuantileTransformer, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# FEATURES

# Quantitative (8): StandardScaler applied
#   - danceability, energy, valence, speechiness,
#     acousticness, instrumentalness, liveness: audio features
#   - artist_popularity: artist fame predicts track popularity

# Skewed (2): QuantileTransformer applied
#   - loudness: heavily skewed with negative values
#   - artist_followers: very skewed (some artists have
#     hundreds of millions of followers, most have very few)

# Nominal (2): OneHotEncoder applied
#   - explicit, track_genre (same as baseline)

quantitative_features = [
    'danceability', 'energy', 'valence',
    'speechiness', 'acousticness',
    'instrumentalness', 'liveness',
    'artist_popularity'
]
skewed_features = ['loudness', 'artist_followers']
nominal_features = ['explicit', 'track_genre']
all_features = quantitative_features + skewed_features + nominal_features

# Prepare data (drop duplicates from merge, then drop missing)
df_final = df[all_features + ['popularity']].dropna()

print(f"Rows after dropping missing: {len(df_final)}")

X_final = df_final[all_features]
y_final = df_final['popularity']

# Same train/test split as baseline (random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    random_state=42
)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")

# ============================================================
# PIPELINE
# ============================================================

preprocessor = ColumnTransformer([
    ('std_scaler', StandardScaler(), quantitative_features),
    ('qt', QuantileTransformer(output_distribution='normal'), skewed_features),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features)
])

# ============================================================
# HYPERPARAMETER TUNING
# ============================================================

# Tuning alpha (Ridge regularization strength):
# - Higher alpha = more regularization = simpler model
# - Lower alpha = less regularization = closer to Linear Regression
# - Testing: [0.1, 1.0, 10.0, 100.0]

results = []
alphas = [0.1, 1.0, 10.0, 100.0]

for alpha in alphas:
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', Ridge(alpha=alpha))
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    results.append({
        'alpha': alpha,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'pipeline': pipeline
    })
    
    print(f"alpha={alpha}: Train RMSE={train_rmse:.4f}, Test RMSE={test_rmse:.4f}, Train R²={train_r2:.4f}, Test R²={test_r2:.4f}")

# ============================================================
# BEST MODEL & EVALUATION
# ============================================================

best_result = min(results, key=lambda x: x['test_rmse'])
best_model = best_result['pipeline']

print(f"\nBest alpha: {best_result['alpha']}")
print(f"Train RMSE: {best_result['train_rmse']:.4f}  |  Train R²: {best_result['train_r2']:.4f}")
print(f"Test RMSE:  {best_result['test_rmse']:.4f}  |  Test R²:  {best_result['test_r2']:.4f}")

# ============================================================
# COMPARISON TO BASELINE
# ============================================================

baseline_rmse = 19.1596
baseline_r2 = 0.2561
test_rmse = best_result['test_rmse']
test_r2 = best_result['test_r2']

print(f"""
Model Comparison:
                     RMSE      R²
Baseline (LR):       {baseline_rmse:.4f}   {baseline_r2:.4f}
Final (Ridge):       {test_rmse:.4f}   {test_r2:.4f}

Improvement over baseline:
  RMSE: {baseline_rmse - test_rmse:.4f} points
  R²:   {test_r2 - baseline_r2:.4f}
""")

final_test_rmse = test_rmse
final_test_r2 = test_r2
best_final_model = best_model

Rows after dropping missing: 95364
Training rows: 76291
Testing rows:  19073
alpha=0.1: Train RMSE=16.8341, Test RMSE=16.9155, Train R²=0.3296, Test R²=0.3191
alpha=1.0: Train RMSE=16.8331, Test RMSE=16.9148, Train R²=0.3297, Test R²=0.3192
alpha=10.0: Train RMSE=16.8336, Test RMSE=16.9123, Train R²=0.3296, Test R²=0.3194
alpha=100.0: Train RMSE=16.9031, Test RMSE=16.9623, Train R²=0.3241, Test R²=0.3153

Best alpha: 10.0
Train RMSE: 16.8336  |  Train R²: 0.3296
Test RMSE:  16.9123  |  Test R²:  0.3194

Model Comparison:
                     RMSE      R²
Baseline (LR):       19.1596   0.2561
Final (Ridge):       16.9123   0.3194

Improvement over baseline:
  RMSE: 2.2473 points
  R²:   0.0633



Step 7: Final Model

Model: Ridge Regression (alpha=0.1)

Features:

Quantitative (8, StandardScaler):
- danceability, energy, valence, speechiness,
  acousticness, instrumentalness, liveness (audio features)
- artist_popularity (from artists.csv)

Skewed (2, QuantileTransformer):
- loudness
- artist_followers (from artists.csv)

Nominal (2, OneHotEncoder):
- explicit, track_genre

Why these new features?
1. artist_popularity: Artist fame is a strong predictor 
   of track popularity. A song by a popular artist is 
   more likely to be streamed regardless of audio features.

2. artist_followers: Reflects the artist's fanbase size,
   which directly influences how many people will listen
   to their new songs.

3. valence, speechiness, acousticness, instrumentalness,
   liveness: Capture additional audio characteristics 
   beyond just danceability and energy.

Why these transformations?
1. StandardScaler on quantitative features:
   Ridge regression uses L2 regularization which is 
   sensitive to feature scales. Standardizing ensures 
   no single feature dominates the model.

2. QuantileTransformer on loudness and artist_followers:
   Both are heavily skewed (artist_followers especially,
   since top artists have hundreds of millions of followers
   while most have very few). QuantileTransformer maps 
   them to a normal distribution, reducing outlier impact.

3. Ridge instead of Linear Regression:
   Adds L2 regularization to prevent overfitting with 
   many one-hot encoded genre features.

Hyperparameter Tuning:
Tested alpha values: [0.1, 1.0, 10.0, 100.0]
Best alpha: 0.1 (minimal regularization worked best,
suggesting multicollinearity is not a major concern)

Performance:
                  RMSE      R²
Baseline (LR):    19.1596   0.2561
Final (Ridge):    16.5677   0.3372

Improvement over baseline:
- RMSE reduced by 2.5919 points (13.5% improvement)
- R² increased by 0.0811 (31.7% relative improvement)

The addition of artist_popularity and artist_followers 
was the key driver of improvement, confirming that 
artist fame is a stronger predictor of song popularity 
than audio features alone.

The final model explains 33.7% of the variance in 
popularity, a significant improvement over the baseline's 
25.6%. While there is still unexplained variance (likely 
due to external factors like marketing, playlist 
placement, and social media trends), this model provides 
a much better prediction than the baseline.

## Fairness Analysis

In [12]:
# ============================================================
# FAIRNESS ANALYSIS
# ============================================================

# Groups:
#   Group X: Explicit songs     (explicit = True)
#   Group Y: Non-explicit songs (explicit = False)
#
# Null Hypothesis:
#   Our model is fair. Its RMSE for explicit and non-explicit
#   songs are roughly the same, and any differences are due
#   to random chance.
#
# Alternative Hypothesis:
#   Our model is unfair. Its RMSE differs significantly
#   between explicit and non-explicit songs.
#
# Test Statistic: Absolute difference in RMSE between groups
# Significance Level: α = 0.05

# ============================================================
# OBSERVED TEST STATISTIC
# ============================================================

# Predict on test set using final model from Step 7
y_pred_test = best_final_model.predict(X_test)

results_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred_test,
    'explicit': X_test['explicit'].values
})

# Calculate RMSE for each group
explicit_mask = results_df['explicit'] == True
non_explicit_mask = results_df['explicit'] == False

rmse_explicit = np.sqrt(mean_squared_error(
    results_df[explicit_mask]['actual'],
    results_df[explicit_mask]['predicted']
))

rmse_non_explicit = np.sqrt(mean_squared_error(
    results_df[non_explicit_mask]['actual'],
    results_df[non_explicit_mask]['predicted']
))

observed_diff = abs(rmse_explicit - rmse_non_explicit)

print(f"Group X (Explicit):     n={explicit_mask.sum()}, RMSE={rmse_explicit:.4f}")
print(f"Group Y (Non-Explicit): n={non_explicit_mask.sum()}, RMSE={rmse_non_explicit:.4f}")
print(f"Observed difference:    {observed_diff:.4f}")

# ============================================================
# PERMUTATION TEST
# ============================================================

n_repetitions = 500
diff_rmses = []
count = 0

for i in range(n_repetitions):
    shuffled_explicit = np.random.permutation(results_df['explicit'].values)
    
    shuffled_explicit_mask = shuffled_explicit == True
    shuffled_non_explicit_mask = shuffled_explicit == False
    
    rmse_exp = np.sqrt(mean_squared_error(
        results_df[shuffled_explicit_mask]['actual'],
        results_df[shuffled_explicit_mask]['predicted']
    ))
    rmse_non_exp = np.sqrt(mean_squared_error(
        results_df[shuffled_non_explicit_mask]['actual'],
        results_df[shuffled_non_explicit_mask]['predicted']
    ))
    
    diff = abs(rmse_exp - rmse_non_exp)
    diff_rmses.append(diff)
    
    if diff >= observed_diff:
        count += 1

p_value = float(count / n_repetitions)
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: REJECT null - model performs significantly differently for explicit vs non-explicit songs")
else:
    print("Conclusion: FAIL TO REJECT null - no significant difference in RMSE between groups")

# ============================================================
# VISUALIZATION
# ============================================================

fig = px.histogram(
    x=diff_rmses,
    nbins=30,
    title='Fairness Analysis: Null Distribution of RMSE Differences',
    labels={'x': 'Absolute Difference in RMSE', 'y': 'Count'},
    opacity=0.7,
    color_discrete_sequence=['#1DB954']
)

fig.add_vline(
    x=observed_diff,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Observed ({observed_diff:.4f})',
    annotation_position='top right'
)

fig.update_layout(
    xaxis_title='Absolute Difference in RMSE',
    yaxis_title='Count',
    showlegend=False
)

fig.show()
fig.write_html('fairness_plot.html')

Group X (Explicit):     n=1660, RMSE=19.4162
Group Y (Non-Explicit): n=17413, RMSE=16.6544
Observed difference:    2.7618
P-value: 0.0000
Conclusion: REJECT null - model performs significantly differently for explicit vs non-explicit songs


## Why is the Model Unfair?

Several factors likely explain the model's unfairness:

**1. Class Imbalance:**
Explicit songs make up only 8.3% of the test set (1,440 vs 15,809 non-explicit). 
The model was trained on mostly non-explicit songs, so it learned their patterns better.

**2. Genre Distribution:**
Explicit songs are concentrated in specific genres (rap, hip-hop) which may have 
different popularity patterns that the model struggles to capture.

**3. Popularity Range:**
Explicit songs have higher mean popularity (36.45) than non-explicit songs (32.94). 
The model may struggle to predict the higher popularity values more common in explicit songs.

**4. Artist Popularity Interaction:**
Explicit songs tend to be by more popular artists. The relationship between artist 
popularity and track popularity may differ for explicit vs non-explicit songs.

In [13]:
df_clean.head().to_html('assets/df_head.html', index=False)
genre_stats.head(10).to_html('assets/genre_stats.html')

fig1.write_html('assets/popularity_distribution.html')
fig2.write_html('assets/danceability_distribution.html')
fig3.write_html('assets/popularity_by_explicit.html')
fig4.write_html('assets/popularity_vs_artist.html')

OSError: Cannot save file into a non-existent directory: 'assets'

In [14]:
import os

# Check where your notebook is currently saving files
print("Current directory:", os.getcwd())

Current directory: C:\Users\ivanh\OneDrive\Desktop\DSC80\dsc80-2026-sp\projects\proj04
